In [ ]:
import pandas as pd
import numpy as np

import json
from datetime import time
import ast

In [ ]:
df=pd.read_csv('pizza_muy_sucia.csv')
df.tail()

In [ ]:
df.isnull().sum()

##  Eliminar filas con valores nulos en todas las columnas excepto 'customer_id'

In [ ]:
columnas_a_verificar = df.columns.drop('customer_id', errors='ignore')
df_limpio = df.dropna(subset=columnas_a_verificar)

## Filtrar registros donde 'completed' es True y borrar la columna

In [ ]:
df_limpio = df_limpio[df_limpio['completed'] == True]
df_limpio = df_limpio.drop(columns=['completed'])
df_limpio.head()

## Quitar duplicados

In [ ]:
df_limpio.duplicated().sum()

In [ ]:
df_limpio = df_limpio.drop_duplicates()

## Quitar columnas redundantes

In [ ]:
conteo_unicos = {}
for columna in df_limpio.columns:
    conteo = df_limpio[columna].value_counts().to_dict()
    if len(conteo) <10:
        conteo_unicos[columna] = {
            'valores_unicos': len(conteo),
            'detalle': conteo
        }

print("Conteo de valores únicos por columna con pocos valores únicos:")
for col, datos in conteo_unicos.items():
    print(f"\nColumna: {col}")
    print(f"Valores únicos: {datos['valores_unicos']}")
    print(f"Ejemplo de conteos: {list(datos['detalle'].items())[:5]}")

In [ ]:
df_limpio = df_limpio.drop(columns=['Country'])

In [ ]:
df_limpio

## Quitar fecha actual de order_time

In [ ]:
df_limpio['order_time'] = pd.to_datetime(df_limpio['order_time']).dt.strftime('%H:%M:%S')
df_limpio['order_time']

## A json

In [ ]:
def safe_literal_eval(s):
    try:
        return ast.literal_eval(str(s))  # Convierte la cadena a diccionario
    except (ValueError, SyntaxError):
        return {}  # Si falla, retorna un diccionario vacío

# Función para eliminar NaN y manejar tipos especiales
def remove_nan(obj):
    if isinstance(obj, dict):
        return {k: remove_nan(v) for k, v in obj.items() if v is not None and not (isinstance(v, float) and np.isnan(v))}
    elif isinstance(obj, list):
        return [remove_nan(item) for item in obj if item is not None and not (isinstance(item, float) and np.isnan(item))]
    elif isinstance(obj, pd.Timestamp): 
        return obj.strftime('%Y-%m-%d')  
    elif isinstance(obj, time):  
        return obj.strftime('%H:%M:%S')  
    else:
        return obj

df_limpio["pizza_ingredients"] = df_limpio["pizza_ingredients"].apply(safe_literal_eval)

# Convertir el DataFrame a un diccionario
dict_data = df_limpio.to_dict(orient='records')

# Limpiar los datos (eliminar NaN y manejar tipos especiales)
cleaned_data = [remove_nan(record) for record in dict_data]

# Convertir a JSON
json_data = json.dumps(cleaned_data, indent=4)

# Imprimir el JSON resultante
print(json_data)

In [ ]:
with open('pizza_1.json', 'w') as file:
    file.write(json_data)

print("El archivo JSON ha sido guardado como 'pizza_1.json'.")

In [ ]:
# Subir datos limpios a MongoDB
# cleaned_data ya esta en memoria desde la celda anterior
import os
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv("../DBNOSQL_Proyecto/.env.local")
client = MongoClient(os.getenv("MONGODB_URI"))
db = client["pizzaDB"]

db["menu"].drop()
db["menu"].insert_many(cleaned_data)

print(f"{len(cleaned_data)} documentos insertados en pizzaDB.menu")
client.close()